In [3]:
import torch
import cv2
import numpy as np
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor


# 配置路徑
checkpoint = "C:/Users/admin/Documents/image-seg/sam_yolo/sam2.1_hiera_large.pt"
model_cfg  = "C:/Users/admin/Documents/GitHub/sam2/sam2/configs/sam2.1/sam2.1_hiera_l.yaml"
image_path = "C:/Users/admin/Documents/image-seg/sam_yolo/bed.jpg"
result_image_path = "C:/Users/admin/Documents/image-seg/sam_yolo/simple_result_4.jpg"
result_image_mask_path = "C:/Users/admin/Documents/image-seg/sam_yolo/simple_result_4_mask.png"

# -*- coding: utf-8 -*-

import os
import cv2

from ultralytics import YOLO

# 載入分割模型
model = YOLO("yolov8m-seg.pt")

predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint, "cpu"))

def extract_mask_compare(image_path):
    image_name = os.path.basename(image_path)
    # 推論圖片
    results = model(image_path, conf=0.15)

    # 如果想儲存結果圖：
    box = None
    for result in results:
        for obj in result.summary():
            if obj["name"] == "bed":
                result.save(filename= "results/" + image_name.replace('.jpg', "_output1.jpg"))
                box = obj["box"]
    if box != None:
        input_point = np.array([[(box["x1"] + box["x2"])//2, (box["y1"]+box["y2"])//2]])
        input_label = np.array([1])
        # 載入圖像
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"無法載入圖像: {image_path}")

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        predictor.set_image(image_rgb)
        masks, scores, logits = predictor.predict(
            point_coords=input_point,
            point_labels=input_label,
            multimask_output=True # 會自動選擇分數最高的
        )
        best_mask_idx = np.argmax(scores)
        best_mask = masks[best_mask_idx]
        best_score = scores[best_mask_idx]

        # save the best mask to file
        mask_filename = image_name.replace('.jpg', "_mask.jpg")
        cv2.imwrite("results/" + mask_filename, (best_mask * 255).astype(np.uint8))

img_files = os.listdir("bed-images")
for img_file in img_files:
    if img_file.endswith('.jpg'):
        image_path = os.path.join("bed-images", img_file)
        extract_mask_compare(image_path)
        print(f"Processed {img_file}")


image 1/1 c:\Users\admin\Documents\GitHub\bedSheetFoldingRobot\bed-images\bed1.jpg: 480x640 1 bed, 168.2ms
Speed: 1.3ms preprocess, 168.2ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)
Processed bed1.jpg

image 1/1 c:\Users\admin\Documents\GitHub\bedSheetFoldingRobot\bed-images\bed2.jpg: 480x640 1 bed, 167.1ms
Speed: 1.1ms preprocess, 167.1ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)
Processed bed2.jpg

image 1/1 c:\Users\admin\Documents\GitHub\bedSheetFoldingRobot\bed-images\bed3.jpg: 448x640 1 bed, 140.8ms
Speed: 0.8ms preprocess, 140.8ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)
Processed bed3.jpg


In [40]:
import numpy as np
import matplotlib.pyplot as plt

def plot_half_cylinder_cloth_dataset(r=1.0, H=4.0, img_size=(400, 800), n_theta=120, n_h=180):
    # Half-cylinder parametric mesh (theta: 0 to pi, h: 0 to H)
    theta = np.linspace(0, np.pi, n_theta)
    h = np.linspace(0, H, n_h)
    Theta, Hh = np.meshgrid(theta, h)
    X = r * np.cos(Theta)
    Y = Hh
    Z = r * np.sin(Theta)
    
    # Corners (keypoints: θ=0/H and θ=π/H)
    keypoints = np.array([
        [ r, 0, 0],   # θ=0, h=0
        [ r, H, 0],   # θ=0, h=H
        [-r, 0, 0],   # θ=π, h=0
        [-r, H, 0],   # θ=π, h=H
    ])
    
    # Plot setup (white bg, no grid/axes/ticks)
    fig = plt.figure(figsize=(img_size[1]/100, img_size[0]/100), dpi=100)
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(X, Y, Z, color='dodgerblue', alpha=1.0, linewidth=0, antialiased=True, zorder=1)
    
    # Keypoints in red
    ax.scatter(keypoints[:,0], keypoints[:,1], keypoints[:,2],
               color='red', s=120, edgecolor='black', zorder=2)
    
    # Remove axes, borders, grid, background
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_axis_off()
    fig.patch.set_facecolor('white')
    
    # Adjust view
    ax.view_init(elev=22, azim=-90)
    ax.set_xlim([-1.2*r, 1.2*r])
    ax.set_ylim([-0.1*H, 1.05*H])
    ax.set_zlim([-1.1*r, 1.1*r])
    ax.dist = 8

    plt.subplots_adjust(left=0, right=1, bottom=0, top=1)
    plt.savefig("half_cylinder_cloth_sample.png", dpi=100, bbox_inches='tight', pad_inches=0, transparent=False)
    plt.close(fig)
    print("Saved data sample as 'half_cylinder_cloth_sample.png'")

plot_half_cylinder_cloth_dataset()


Saved data sample as 'half_cylinder_cloth_sample.png'
